<a href="https://colab.research.google.com/github/sakshimohta17/DVJlab_code/blob/main/Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1) Upload your mesh (.msh) in Colab
# ============================================================
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

# Pick the first uploaded .msh automatically
msh_name = [k for k in uploaded.keys() if k.lower().endswith(".msh")]
if not msh_name:
    # helpful message if someone accidentally uploads a zip
    zip_name = [k for k in uploaded.keys() if k.lower().endswith(".zip")]
    if zip_name:
        raise FileNotFoundError(
            f"You uploaded a .zip ({zip_name[0]}). Please upload the raw .msh file instead."
        )
    raise FileNotFoundError("No .msh uploaded. Please upload a .msh mesh file.")

MSH_PATH = "/content/" + msh_name[0]
print("Using MSH_PATH:", MSH_PATH)

# ============================================================
# 2) Install dependencies
# ============================================================
!pip -q install meshio pyvista vtk numpy pandas

# ============================================================
# 3) Imports + settings
# ============================================================
import os
from typing import Optional, Dict

import numpy as np
import pandas as pd
import meshio
import pyvista as pv

OUT_METRICS_CSV = "/content/Z210W0_metrics_summary.csv"
OUT_CURV_CSV = "/content/Z210W0_curvature_summary.csv"

# bottom 20% closest LV↔RV endo points = septal region
SEPTUM_FRACTION = 0.20

# ============================================================
# 4) Helper functions
# ============================================================
def summarize(x: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x).astype(float)
    return {
        "mean": float(np.mean(x)),
        "median": float(np.median(x)),
        "p05": float(np.percentile(x, 5)),
        "p95": float(np.percentile(x, 95)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "n": int(x.size),
    }

def phys_id(mesh: meshio.Mesh, name: str) -> int:
    if name not in mesh.field_data:
        raise KeyError(f"Physical group '{name}' not found. Available: {list(mesh.field_data.keys())}")
    return int(mesh.field_data[name][0])

def make_surface_from_tris(points: np.ndarray, tris: np.ndarray) -> pv.PolyData:
    faces = np.hstack([np.full((tris.shape[0], 1), 3, dtype=np.int64), tris]).ravel()
    return pv.PolyData(points, faces).clean().triangulate()

def extract_surface(mesh: meshio.Mesh, phys: int) -> pv.PolyData:
    tris_all = []
    for cb, phys_block in zip(mesh.cells, mesh.cell_data["gmsh:physical"]):
        if cb.type != "triangle":
            continue
        phys_block = np.asarray(phys_block)
        if np.all(phys_block == phys):
            tris_all.append(cb.data)
        else:
            tris_all.append(cb.data[phys_block == phys])
    tris = np.vstack([t for t in tris_all if t.size > 0])
    if tris.size == 0:
        raise ValueError(f"No triangles found for phys={phys}")
    return make_surface_from_tris(mesh.points, tris)

def pointwise_distance(source: pv.PolyData, target: pv.PolyData) -> np.ndarray:
    s = source.compute_implicit_distance(target)  # adds implicit_distance
    return np.abs(np.asarray(s["implicit_distance"]))

def to_polydata(obj) -> pv.PolyData:
    return obj if isinstance(obj, pv.PolyData) else obj.extract_surface().triangulate().clean()

def close_surface_with_cap(endo: pv.PolyData, cap: Optional[pv.PolyData]) -> pv.PolyData:
    s = endo.clean().triangulate()
    if cap is not None and cap.n_cells > 0:
        s = to_polydata(s.merge(cap.clean().triangulate(), merge_points=True)).clean().triangulate()
    s = s.fill_holes(1e9).clean().triangulate()
    s = s.compute_normals(auto_orient_normals=True, consistent_normals=True)
    return s

# ============================================================
# 5) Read mesh (direct .msh)
# ============================================================
mesh = meshio.read(MSH_PATH)

if "gmsh:physical" not in mesh.cell_data:
    raise ValueError("Mesh has no 'gmsh:physical' cell_data. Cannot extract labeled surfaces.")

print("Physical groups found:", mesh.field_data)

# ============================================================
# 6) Extract LV/RV endo, epi, base surfaces
# ============================================================
lv_id = phys_id(mesh, "lv")
rv_id = phys_id(mesh, "rv")
epi_id = phys_id(mesh, "epi")
base_id = phys_id(mesh, "base")

lv_endo = extract_surface(mesh, lv_id)
rv_endo = extract_surface(mesh, rv_id)
epi = extract_surface(mesh, epi_id)
base = extract_surface(mesh, base_id)

print("Surface cells:",
      "LV endo:", lv_endo.n_cells,
      "RV endo:", rv_endo.n_cells,
      "Epi:", epi.n_cells,
      "Base:", base.n_cells)

# ============================================================
# 7) Split base cap into LV-cap vs RV-cap (automatic)
# ============================================================
base_centers = base.cell_centers().points
centers_pd = pv.PolyData(base_centers)

d_to_lv = np.abs(centers_pd.compute_implicit_distance(lv_endo)["implicit_distance"])
d_to_rv = np.abs(centers_pd.compute_implicit_distance(rv_endo)["implicit_distance"])
assign_lv = d_to_lv <= d_to_rv

lv_base = base.extract_cells(np.where(assign_lv)[0]).clean().triangulate()
rv_base = base.extract_cells(np.where(~assign_lv)[0]).clean().triangulate()

# ============================================================
# 8) Volumes (close endo with base caps)
# ============================================================
lv_closed = close_surface_with_cap(lv_endo, lv_base)
rv_closed = close_surface_with_cap(rv_endo, rv_base)

lv_vol = float(lv_closed.volume)
rv_vol = float(rv_closed.volume)

# ============================================================
# 9) Thickness (endo->epi), Septum (LV↔RV endo distance)
# ============================================================
lv_to_epi = pointwise_distance(lv_endo, epi)
rv_to_epi = pointwise_distance(rv_endo, epi)

# Septal-facing region detection
lv_to_rv = pointwise_distance(lv_endo, rv_endo)
rv_to_lv = pointwise_distance(rv_endo, lv_endo)

lv_sep_thresh = np.quantile(lv_to_rv, SEPTUM_FRACTION)
rv_sep_thresh = np.quantile(rv_to_lv, SEPTUM_FRACTION)

lv_is_septal = lv_to_rv <= lv_sep_thresh
rv_is_septal = rv_to_lv <= rv_sep_thresh

# Free wall thickness excludes septal-facing points
lv_freewall_th = lv_to_epi[~lv_is_septal]
rv_freewall_th = rv_to_epi[~rv_is_septal]

# Septum thickness definition used here:
# LV↔RV endocardium distance in septal-facing regions (both sides)
septum_th = np.concatenate([lv_to_rv[lv_is_septal], rv_to_lv[rv_is_septal]])

# ============================================================
# 10) Curvature (mean + gaussian)
# ============================================================
lv_mean_curv = np.asarray(lv_endo.curvature(curv_type="mean"))
rv_mean_curv = np.asarray(rv_endo.curvature(curv_type="mean"))
epi_mean_curv = np.asarray(epi.curvature(curv_type="mean"))

lv_gauss_curv = np.asarray(lv_endo.curvature(curv_type="gaussian"))
rv_gauss_curv = np.asarray(rv_endo.curvature(curv_type="gaussian"))
epi_gauss_curv = np.asarray(epi.curvature(curv_type="gaussian"))

# ============================================================
# 11) Save CSV outputs
# ============================================================
rows = [
    ("LV endocardium volume", lv_vol),
    ("RV endocardium volume", rv_vol),
]

def add_block(prefix, stats):
    rows.extend([
        (f"{prefix} thickness mean", stats["mean"]),
        (f"{prefix} thickness median", stats["median"]),
        (f"{prefix} thickness p05", stats["p05"]),
        (f"{prefix} thickness p95", stats["p95"]),
        (f"{prefix} thickness min", stats["min"]),
        (f"{prefix} thickness max", stats["max"]),
        (f"{prefix} thickness n_points", stats["n"]),
    ])

add_block("LV free wall", summarize(lv_freewall_th))
add_block("RV free wall", summarize(rv_freewall_th))
add_block("Septum (LV↔RV endo distance)", summarize(septum_th))

df = pd.DataFrame(rows, columns=["metric", "value"])
df.to_csv(OUT_METRICS_CSV, index=False)

curv_rows = []
for name, arr in [
    ("LV_endo_mean", lv_mean_curv),
    ("RV_endo_mean", rv_mean_curv),
    ("Epi_mean", epi_mean_curv),
    ("LV_endo_gaussian", lv_gauss_curv),
    ("RV_endo_gaussian", rv_gauss_curv),
    ("Epi_gaussian", epi_gauss_curv),
]:
    s = summarize(arr)
    curv_rows.append((name, s["mean"], s["median"], s["p05"], s["p95"], s["min"], s["max"], s["n"]))

curv_df = pd.DataFrame(
    curv_rows,
    columns=["surface", "mean", "median", "p05", "p95", "min", "max", "n_points"],
)
curv_df.to_csv(OUT_CURV_CSV, index=False)

# ============================================================
# 12) Print results + download outputs
# ============================================================
print("\n=== RESULTS ===")
print("LV endocardium volume:", lv_vol)
print("RV endocardium volume:", rv_vol)
print("LV free wall thickness stats:", summarize(lv_freewall_th))
print("RV free wall thickness stats:", summarize(rv_freewall_th))
print(f"Septum thickness stats (LV↔RV endo distance; bottom {int(SEPTUM_FRACTION*100)}% septal-facing):",
      summarize(septum_th))

print("\nSaved files in /content:")
print(" -", OUT_METRICS_CSV)
print(" -", OUT_CURV_CSV)

files.download(OUT_METRICS_CSV)
files.download(OUT_CURV_CSV)


Saving W283W12.msh to W283W12 (1).msh
Uploaded: ['W283W12 (1).msh']
Using MSH_PATH: /content/W283W12 (1).msh

Physical groups found: {'rv': array([1, 2]), 'lv': array([2, 2]), 'epi': array([3, 2]), 'base': array([4, 2]), 'Group_1': array([5, 3])}
Surface cells: LV endo: 2445 RV endo: 2153 Epi: 7226 Base: 1094

=== RESULTS ===
LV endocardium volume: 256.8433833020353
RV endocardium volume: 124.52829081756578
LV free wall thickness stats: {'mean': 2.4507950147271047, 'median': 2.1747889812597028, 'p05': 1.6838715315069575, 'p95': 4.173730374155154, 'min': 1.5603781237667513, 'max': 5.581919525332246, 'n': 1000}
RV free wall thickness stats: {'mean': 1.8634425641277124, 'median': 1.5185377977302532, 'p05': 1.1555545804614915, 'p95': 3.832618188943021, 'min': 0.5678693963098155, 'max': 4.6474875985492945, 'n': 887}
Septum thickness stats (LV↔RV endo distance; bottom 20% septal-facing): {'mean': 1.386494725603156, 'median': 1.4123421951309536, 'p05': 1.0134554848923236, 'p95': 1.74123206967

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>